# Fine-tuning FLAN-T5-XL para Text-to-SQL con Dataset CSV

Entrenamiento de FLAN-T5-XL (3B) para generar consultas SQL con cuantización 8-bit + LoRA usando dataset CSV.

**Configuración optimizada para Tesla T4:**
- Cuantización 8-bit CRÍTICA para 3B parámetros
- LoRA para fine-tuning eficiente
- Configuración de memoria conservadora
- Dataset CSV pre-limpiado y estratificado

In [ ]:
# Autenticación en Hugging Face
from huggingface_hub import login
login()

In [ ]:
# Instalar dependencias necesarias
!pip install -q bitsandbytes accelerate
print("✅ Dependencias instaladas")

In [ ]:
# Imports
import torch
import torch.nn as nn
import os
import json
import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)

from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    prepare_model_for_kbit_training,
    PeftModel
)

from datasets import Dataset as HFDataset

print(f"✅ PyTorch: {torch.__version__}")
print(f"🎮 CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"🎯 GPU: {props.name} ({props.total_memory / 1024**3:.1f}GB)")

In [ ]:
# Configuración
CONFIG = {
    "model_name": "google/flan-t5-xl",
    "csv_file_path": "C:/Users/arasa/OneDrive - UTN - Santa Fe/Facultad/5° Año/Proyecto Final/proyecto-final-repo/proyecto_final/inputs/dataset_estratificado_5000.csv",
    "max_samples": 10000,  # Conservador para Tesla T4
    "max_input_length": 512,
    "max_target_length": 256,
    "batch_size": 1,
    "gradient_accumulation": 4,  # Batch efectivo = 4
    "learning_rate": 2e-5,  # Conservador para 3B
    "num_epochs": 1,  # Una época para prueba
    "warmup_ratio": 0.1,
    "save_steps": 200,        # Guardar más frecuente
    "eval_steps": 200,
    "eval_strategy": "steps",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "save_strategy": "epoch",
    "logging_steps": 10,
    "output_dir": "C:/Users/arasa/OneDrive - UTN - Santa Fe/Facultad/5° Año/Proyecto Final/proyecto-final-repo/proyecto_final/outputs/flan-t5-xl-sql-csv"
}

LORA_CONFIG = {
    "r": 4,  # Rank bajo para memoria
    "lora_alpha": 16,
    "target_modules": ["q"],  # Solo query para memoria
    "lora_dropout": 0.1,
    "bias": "none",
    "task_type": TaskType.SEQ_2_SEQ_LM
}

print("📋 Configuración:")
print(f"   Modelo: {CONFIG['model_name']}")
print(f"   CSV: {CONFIG['csv_file_path']}")
print(f"   Batch efectivo: {CONFIG['batch_size']} × {CONFIG['gradient_accumulation']} = {CONFIG['batch_size'] * CONFIG['gradient_accumulation']}")
print(f"   LoRA rank: {LORA_CONFIG['r']}")
print(f"   Max muestras: {CONFIG['max_samples']}")

In [ ]:
# Preparar datos desde CSV
def preparar_datos():
    print("📊 Cargando dataset desde CSV...")
    
    # Verificar que el archivo existe
    if not os.path.exists(CONFIG["csv_file_path"]):
        raise FileNotFoundError(f"No se encontró el archivo CSV: {CONFIG['csv_file_path']}")
    
    # Cargar CSV
    df = pd.read_csv(CONFIG["csv_file_path"])
    
    print(f"📦 Dataset cargado: {len(df)} registros totales")
    print(f"📝 Columnas disponibles: {list(df.columns)}")
    
    # Verificar columnas necesarias
    required_columns = ['sql_prompt', 'sql_context', 'sql']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Faltan columnas requeridas: {missing_columns}")
    
    # Filtrar solo registros de entrenamiento (si existe columna split)
    if 'split' in df.columns:
        train_df = df[df['split'] == 'train'].copy()
        print(f"🚂 Registros de entrenamiento: {len(train_df)}")
    else:
        train_df = df.copy()
        print(f"📊 Usando todos los registros para entrenamiento: {len(train_df)}")
    
    # Verificar que las columnas necesarias no tengan valores nulos
    for col in required_columns:
        null_count = train_df[col].isnull().sum()
        if null_count > 0:
            print(f"⚠️  Removiendo {null_count} registros con valores nulos en '{col}'")
            train_df = train_df[train_df[col].notnull()]
    
    # Limitar muestras si es necesario
    if len(train_df) > CONFIG["max_samples"]:
        print(f"✂️  Limitando a {CONFIG['max_samples']} muestras")
        train_df = train_df.head(CONFIG["max_samples"]).reset_index(drop=True)
    
    print(f"✅ Dataset final: {len(train_df)} ejemplos")
    
    # Mostrar estadísticas del dataset
    if 'sql_complexity' in train_df.columns:
        print(f"📈 Distribución por complejidad:")
        complexity_counts = train_df['sql_complexity'].value_counts()
        for complexity, count in complexity_counts.items():
            print(f"   {complexity}: {count} ({count/len(train_df)*100:.1f}%)")
    
    if 'domain' in train_df.columns:
        print(f"🏷️  Dominios únicos: {train_df['domain'].nunique()}")
    
    return train_df

def formatear_datos(sql_context, sql_prompt, sql):
    input_text = f"""Based on the database schema, write a SQL query.

Schema: {sql_context}
Question: {sql_prompt}
SQL:"""
    
    target_text = sql.strip()
    if not target_text.endswith(';'):
        target_text += ';'
    
    return input_text, target_text

# Crear datasets
df_sql = preparar_datos()

inputs = []
targets = []

print("🔄 Formateando datos...")
for _, row in df_sql.iterrows():
    input_text, target_text = formatear_datos(
        row['sql_context'], row['sql_prompt'], row['sql']
    )
    inputs.append(input_text)
    targets.append(target_text)

# Split train/eval 90/10
split_idx = int(len(inputs) * 0.9)
train_data = {"input_text": inputs[:split_idx], "target_text": targets[:split_idx]}
eval_data = {"input_text": inputs[split_idx:], "target_text": targets[split_idx:]}

train_dataset = HFDataset.from_dict(train_data)
eval_dataset = HFDataset.from_dict(eval_data)

print(f"📦 Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# Mostrar ejemplo
print("\n📝 Ejemplo de formateo:")
print(f"Input: {inputs[0][:200]}...")
print(f"Target: {targets[0]}")

In [ ]:
# Cargar modelo con cuantización 8-bit
def cargar_modelo():
    print("🧠 Cargando modelo FLAN-T5-XL con cuantización 8-bit...")
    
    # Configuración de cuantización
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16,
        bnb_8bit_use_double_quant=False  # Deshabilitar doble cuantización para estabilidad
    )
    
    # Cargar modelo
    model = T5ForConditionalGeneration.from_pretrained(
        CONFIG["model_name"],
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
    
    # Preparar modelo para k-bit training (CRÍTICO para LoRA + cuantización)
    model = prepare_model_for_kbit_training(model)
    
    # Cargar tokenizer
    tokenizer = T5Tokenizer.from_pretrained(CONFIG["model_name"])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Mostrar uso de memoria
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        print(f"💾 Memoria GPU: {allocated:.1f}GB")
    
    print("✅ Modelo cargado con cuantización 8-bit")
    return model, tokenizer

base_model, tokenizer = cargar_modelo()

In [ ]:
# Tokenizar datos
def tokenizar_datos(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=CONFIG["max_input_length"],
        truncation=True,
        padding=False
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["target_text"],
            max_length=CONFIG["max_target_length"],
            truncation=True,
            padding=False
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("🔤 Tokenizando datasets...")
train_dataset = train_dataset.map(tokenizar_datos, batched=True, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(tokenizar_datos, batched=True, remove_columns=eval_dataset.column_names)

print(f"✅ Tokenización completada: {len(train_dataset)} train, {len(eval_dataset)} eval")

In [ ]:
# Aplicar LoRA
def aplicar_lora(model):
    print("🔧 Aplicando LoRA...")
    
    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        target_modules=LORA_CONFIG["target_modules"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"]
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    return model

model = aplicar_lora(base_model)

# Crear data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)

In [ ]:
# Configurar entrenamiento
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    logging_steps=CONFIG["logging_steps"],
    
    # Configuración simplificada para cuantización + LoRA
    optim="adamw_torch",  # Optimizador estándar
    fp16=False,  # Sin FP16 con cuantización 8-bit
    gradient_checkpointing=False,  # Sin gradient checkpointing
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    
    # Evaluacion y guardado
    eval_strategy=CONFIG["eval_strategy"],
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=1,
    load_best_model_at_end=CONFIG["load_best_model_at_end"],
    metric_for_best_model=CONFIG["metric_for_best_model"],
    
    remove_unused_columns=False,
    report_to="none",
    seed=42
)

# Crear trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("✅ Trainer configurado")

In [ ]:
# Entrenar modelo
def entrenar():
    print("🚀 Iniciando entrenamiento...")
    
    # Verificar parámetros entrenables
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"📊 Parámetros entrenables: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    
    if trainable_params == 0:
        raise RuntimeError("❌ No hay parámetros entrenables - problema con LoRA")
    
    # Limpiar memoria
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    
    try:
        # Entrenar
        result = trainer.train()
        
        print(f"\n🎉 Entrenamiento completado!")
        print(f"📉 Loss final: {result.training_loss:.4f}")
        
        # Guardar modelo
        print("💾 Guardando modelo...")
        os.makedirs(CONFIG["output_dir"], exist_ok=True)
        model.save_pretrained(CONFIG["output_dir"])
        tokenizer.save_pretrained(CONFIG["output_dir"])
        
        # Guardar info
        model_info = {
            "base_model": CONFIG["model_name"],
            "csv_file": CONFIG["csv_file_path"],
            "lora_config": LORA_CONFIG,
            "training_config": CONFIG,
            "final_loss": result.training_loss,
            "trainable_params": trainable_params,
            "total_params": total_params,
            "dataset_size": len(train_dataset)
        }
        
        with open(f"{CONFIG['output_dir']}/model_info.json", "w") as f:
            json.dump(model_info, f, indent=2)
        
        print(f"✅ Modelo guardado en: {CONFIG['output_dir']}")
        return True
        
    except Exception as e:
        print(f"❌ Error durante entrenamiento: {e}")
        return False
    
    finally:
        # Limpieza final
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

# Ejecutar entrenamiento
success = entrenar()

if success:
    print("\n🎉 ¡Entrenamiento exitoso!")
else:
    print("\n❌ Entrenamiento falló")

In [ ]:
# Cargar modelo entrenado desde disco
def cargar_modelo_entrenado():
    print("🔄 Cargando modelo entrenado desde disco...")
    
    # Verificar que existe el modelo guardado
    if not os.path.exists(CONFIG['output_dir']):
        print(f"❌ No se encontró el modelo en: {CONFIG['output_dir']}")
        return None, None
    
    try:
        # Configuración de cuantización (igual que el entrenamiento)
        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_compute_dtype=torch.float16,
            bnb_8bit_use_double_quant=False
        )
        
        # Cargar modelo base con cuantización
        print("📥 Cargando modelo base...")
        base_model = T5ForConditionalGeneration.from_pretrained(
            CONFIG["model_name"],
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16
        )
        
        # Preparar para k-bit training
        base_model = prepare_model_for_kbit_training(base_model)
        
        # Cargar adaptadores LoRA entrenados
        print("🔧 Cargando adaptadores LoRA entrenados...")
        trained_model = PeftModel.from_pretrained(base_model, CONFIG["output_dir"])
        
        # Cargar tokenizer
        tokenizer = T5Tokenizer.from_pretrained(CONFIG["output_dir"])
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Verificar memoria
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated(0) / 1024**3
            print(f"💾 Memoria GPU: {allocated:.1f}GB")
        
        print("✅ Modelo entrenado cargado correctamente")
        return trained_model, tokenizer
        
    except Exception as e:
        print(f"❌ Error cargando modelo entrenado: {e}")
        return None, None

In [ ]:
# Evaluación completa con ejemplos del dataset CSV
def evaluar_modelo_con_dataset(num_ejemplos=10):
    print(f"🔬 Evaluando modelo con {num_ejemplos} ejemplos del dataset CSV...")
    
    # Cargar modelo entrenado
    trained_model, trained_tokenizer = cargar_modelo_entrenado()
    
    if trained_model is None:
        print("❌ No se pudo cargar el modelo entrenado")
        return
    
    # Usar ejemplos del dataset CSV
    eval_examples = df_sql.tail(num_ejemplos).reset_index(drop=True)
    
    print("=" * 80)
    print("🧪 EVALUACIÓN DEL MODELO ENTRENADO CON DATASET CSV")
    print("=" * 80)
    
    for i, row in eval_examples.iterrows():
        print(f"\n📊 EJEMPLO {i+1}/{num_ejemplos}")
        print("-" * 60)
        
        # Preparar input
        test_context = row['sql_context']
        test_prompt = row['sql_prompt']
        expected_sql = row['sql']
        
        # Mostrar información adicional si está disponible
        if 'domain' in row:
            print(f"🏷️  Dominio: {row['domain']}")
        if 'sql_complexity' in row:
            print(f"⚡ Complejidad: {row['sql_complexity']}")
        if 'sql_task_type' in row:
            print(f"🎯 Tipo de tarea: {row['sql_task_type']}")
        
        input_text = f"""Based on the database schema, write a SQL query.

Schema: {test_context}
Question: {test_prompt}
SQL:"""
        
        # Generar predicción
        try:
            inputs = trained_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(trained_model.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = trained_model.generate(
                    **inputs,
                    max_new_tokens=128,
                    num_beams=3,
                    early_stopping=True,
                    pad_token_id=trained_tokenizer.pad_token_id,
                    do_sample=False
                )
            
            generated_sql = trained_tokenizer.decode(outputs[0], skip_special_tokens=True)
            # Limpiar salida
            predicted_sql = generated_sql.replace(input_text, "").strip()
            
            # Mostrar resultados
            print(f"🏗️  Esquema: {test_context[:80]}{'...' if len(test_context) > 80 else ''}")
            print(f"❓ Pregunta: {test_prompt}")
            print(f"✅ SQL Esperado: {expected_sql}")
            print(f"🤖 SQL Generado: {predicted_sql}")
            
            # Análisis básico de similitud
            if predicted_sql.lower() in expected_sql.lower() or expected_sql.lower() in predicted_sql.lower():
                print("🎯 Estado: ✅ SIMILAR")
            elif any(keyword in predicted_sql.upper() for keyword in ['SELECT', 'FROM', 'WHERE', 'INSERT', 'UPDATE', 'DELETE']):
                print("🎯 Estado: ⚠️  SQL VÁLIDO")
            else:
                print("🎯 Estado: ❌ PROBLEMÁTICO")
                
        except Exception as e:
            print(f"❌ Error generando SQL: {e}")
            print(f"✅ SQL Esperado: {expected_sql}")
            print(f"🤖 SQL Generado: ERROR")
            print("🎯 Estado: ❌ ERROR")
    
    print("\n" + "=" * 80)
    print("🏁 EVALUACIÓN COMPLETADA")
    print("=" * 80)

# Ejecutar evaluación completa
evaluar_modelo_con_dataset(30)